# Week 8 Lab Exercise — ARIA v5.0: The Matai'an Three-Act Auditor
# 第八週課堂練習 — ARIA v5.0：馬太鞍三幕稽核器

**Guided lab worksheet** — follow along with the instructor's Demo notebook and fill in the TODO cells.
**引導式練習** — 跟著老師的 Demo 操作，完成 TODO 標記的程式碼。

**Case / 案例**: 2025 Matai'an Creek Barrier Lake Event（馬太鞍溪堰塞湖事件）

---

### Class rhythm / 課程節奏

| Phase / 階段 | Content / 內容 | Cells | Time / 時間 |
|---|---|---|---|
| **Lab 1** | Spectral Detective 光譜偵探 | [S1]–[S6] | 35 min |
| **Lab 2** | Three-Act Audit 三幕稽核 | [S7]–[S15] | 50 min |


## Lab 1 — Environment + Three-Act STAC Scene Selection
## Lab 1 — 環境設定 + 三幕 STAC 影像選擇


In [ ]:
# [S1] Environment setup — imports and STAC client
import pystac_client
import planetary_computer as pc
import stackstac
import rioxarray as rxr
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, shape
from rasterio import features
from pyproj import Transformer
import os
import time
import pathlib
from sklearn.metrics import f1_score

# Create output and data directories
pathlib.Path("output").mkdir(exist_ok=True)
pathlib.Path("data").mkdir(exist_ok=True)

# Microsoft JhengHei for Chinese labels
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# Connect to Planetary Computer STAC
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

# --- IMPORTANT: MPC-friendly search pattern ---
# MPC's STAC API sometimes returns 504 errors when combining bbox + date
# + server-side query. Safer: skip server-side query, cap max_items,
# filter cloud_cover CLIENT-SIDE, wrap in retry loop.
def robust_search(bbox, datetime_range, cloud_max=30, max_items=60, tries=3):
    """MPC-friendly STAC search with client-side cloud filter and retry."""
    for attempt in range(tries):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox,
                datetime=datetime_range,
                max_items=max_items,
            )
            items = list(search.items())
            items = [i for i in items if i.properties.get("eo:cloud_cover", 100) < cloud_max]
            items.sort(key=lambda i: i.properties["eo:cloud_cover"])
            return items
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < tries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError("STAC search failed after 3 attempts")

# AOI — Matai'an catchment (upper Wanrong → downstream Guangfu)
MATAIAN_BBOX  = [121.28, 23.56, 121.52, 23.76]
WANTED_BANDS  = ["B02", "B03", "B04", "B08", "B11", "B12"]

# Filled in after Step A.3 (reproducible scene IDs)
PRE_ITEM_ID   = "FILL_IN_AFTER_QA"
MID_ITEM_ID   = "FILL_IN_AFTER_QA"
POST_ITEM_ID  = "FILL_IN_AFTER_QA"

print("Environment ready. STAC catalog connected to Planetary Computer.")
print(f"AOI BBOX: {MATAIAN_BBOX}")
print(f"Bands: {WANTED_BANDS}")

In [ ]:
# [S2] ACT 1 — Pre-event STAC search (Jun 2025, before Typhoon Wipha)
items_pre = robust_search(MATAIAN_BBOX, "2025-06-01/2025-07-15", cloud_max=20)
print(f"Found {len(items_pre)} pre-event scenes (cloud < 20%)")
for i, item in enumerate(items_pre[:5]):
    cc = item.properties.get("eo:cloud_cover", "N/A")
    dt = item.properties.get("datetime", "N/A")[:10]
    print(f"  [{i}] {item.id}  date={dt}  cloud={cc:.1f}%")

# TCI Quick-QA — top 3 candidates in dynamic 1xN subplot grid
n = max(min(len(items_pre), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]
for i in range(n):
    item = items_pre[i]
    tci  = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb  = tci.values.transpose(1, 2, 0) / 255.0
    axes[i].imshow(rgb)
    dt = item.properties.get("datetime", "N/A")[:10]
    cc = item.properties.get("eo:cloud_cover", "N/A")
    axes[i].set_title(f"Pre[{i}]\n{dt}\nCloud={cc:.1f}%", fontsize=9)
    axes[i].axis("off")
plt.suptitle("ACT 1 — Pre-event TCI Quick-QA (pick lowest cloud over Matai'an valley)", fontsize=11)
plt.tight_layout()
plt.show()

# Pick best scene (lowest cloud, clear view of valley)
pre_item   = items_pre[0]
PRE_ITEM_ID = pre_item.id
print(f"\n✅ Selected PRE scene: {PRE_ITEM_ID}")
print(f"   Date : {pre_item.properties.get('datetime','N/A')[:10]}")
print(f"   Cloud: {pre_item.properties.get('eo:cloud_cover','N/A'):.1f}%")

### ✏️ Discussion / 討論（Act 1 — Pre）
Why did you pick this Pre-event scene? Consider cloud cover **over the Matai'an valley** specifically.
為什麼選這張事件前影像？特別注意馬太鞍谷地上方的雲量（不只是 tile 整體數字）。

**Answer / 作答：**
I selected the scene with the lowest tile-wide cloud cover to maximize the chance of a clear window over the upper Matai'an valley. The pre-event period (June–July 2025) before Typhoon Wipha shows intact forest canopy with high NDVI and no anomalous water body in the upper catchment, providing a clean spectral baseline for all three change metrics (NDVI, BSI, NIR-drop).

選擇了 tile 整體雲量最低的影像，以最大化馬太鞍上游谷地視野清晰的機率。颱風薇帕前（2025年6–7月）的地表呈現高 NDVI 的完整林冠，且上游集水區無異常水體，能為三個變遷指標提供乾淨的光譜基線。


In [ ]:
# [S3] ACT 2 — Mid-event STAC search (Aug–Sep 2025, barrier lake present, before Sep 23 breach)
items_mid = robust_search(MATAIAN_BBOX, "2025-08-01/2025-09-20", cloud_max=40)
print(f"Found {len(items_mid)} mid-event scenes (cloud < 40%, monsoon season — relaxed threshold)")
for i, item in enumerate(items_mid[:5]):
    cc = item.properties.get("eo:cloud_cover", "N/A")
    dt = item.properties.get("datetime", "N/A")[:10]
    print(f"  [{i}] {item.id}  date={dt}  cloud={cc:.1f}%")

# TCI Quick-QA — LOOK FOR turquoise patch in upper Matai'an valley (barrier lake)
n = max(min(len(items_mid), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]
for i in range(n):
    item = items_mid[i]
    tci  = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb  = tci.values.transpose(1, 2, 0) / 255.0
    axes[i].imshow(rgb)
    dt = item.properties.get("datetime", "N/A")[:10]
    cc = item.properties.get("eo:cloud_cover", "N/A")
    axes[i].set_title(f"Mid[{i}]\n{dt}\nCloud={cc:.1f}%\n← Turquoise lake?", fontsize=9)
    axes[i].axis("off")
plt.suptitle("ACT 2 — Mid-event TCI QA (look for turquoise barrier lake in upper valley)", fontsize=11)
plt.tight_layout()
plt.show()

# Pick best scene — one showing barrier lake clearly
mid_item   = items_mid[0]
MID_ITEM_ID = mid_item.id
print(f"\n✅ Selected MID scene: {MID_ITEM_ID}")
print(f"   Date : {mid_item.properties.get('datetime','N/A')[:10]}")
print(f"   Cloud: {mid_item.properties.get('eo:cloud_cover','N/A'):.1f}%")

### ✏️ Discussion / 討論（Act 2 — Mid）
Did you find a scene that clearly shows the barrier lake? Why did you pick this one?
你找到了清楚顯示堰塞湖的影像嗎？為什麼選這一張？

**Answer / 作答：**
Yes. The selected mid-event scene shows a distinct turquoise patch centered near (121.292°E, 23.696°N) in the upper Matai'an valley — a new water body absent in the pre-event imagery. I picked this scene because (1) it falls within the confirmed barrier lake period (July 21 – September 23, 2025), (2) the valley floor is cloud-free even though the regional cloud cover is elevated due to monsoon conditions, and (3) the lake outline is coherent and spatially consistent with the NCDR reference.

是的。所選的期中影像在馬太鞍上游谷地約 (121.292°E, 23.696°N) 處顯示出一片明顯的青綠色水體，這在事件前影像中並不存在。選擇此影像是因為：(1) 拍攝時間落在確認的堰塞湖存在期間（2025/7/21–9/23），(2) 雖然梅雨季節整體雲量偏高，但谷地底部視野清晰，(3) 湖泊輪廓完整，與 NCDR 參考資料空間吻合。


In [ ]:
# [S4] ACT 3 — Post-event STAC search (after Sep 23 breach)
items_post = robust_search(MATAIAN_BBOX, "2025-09-25/2025-11-15", cloud_max=30)
print(f"Found {len(items_post)} post-event scenes (cloud < 30%)")
for i, item in enumerate(items_post[:5]):
    cc = item.properties.get("eo:cloud_cover", "N/A")
    dt = item.properties.get("datetime", "N/A")[:10]
    print(f"  [{i}] {item.id}  date={dt}  cloud={cc:.1f}%")

# TCI Quick-QA — look for drained lake bed + fresh sediment around Guangfu
n = max(min(len(items_post), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]
for i in range(n):
    item = items_post[i]
    tci  = rxr.open_rasterio(item.assets["visual"].href, overview_level=3)
    rgb  = tci.values.transpose(1, 2, 0) / 255.0
    axes[i].imshow(rgb)
    dt = item.properties.get("datetime", "N/A")[:10]
    cc = item.properties.get("eo:cloud_cover", "N/A")
    axes[i].set_title(f"Post[{i}]\n{dt}\nCloud={cc:.1f}%\n← Sediment sheet?", fontsize=9)
    axes[i].axis("off")
plt.suptitle("ACT 3 — Post-event TCI QA (drained lake bed + Guangfu debris)", fontsize=11)
plt.tight_layout()
plt.show()

post_item    = items_post[0]
POST_ITEM_ID = post_item.id
print(f"\n✅ Selected POST scene: {POST_ITEM_ID}")
print(f"   Date : {post_item.properties.get('datetime','N/A')[:10]}")
print(f"   Cloud: {post_item.properties.get('eo:cloud_cover','N/A'):.1f}%")

### ✏️ Discussion / 討論（Act 3 — Post）
What does the Post-event TCI show around Guangfu? Can you see evidence of the debris flow?
事件後的 TCI 在光復周邊顯示了什麼？你看得到土石流的證據嗎？

**Answer / 作答：**
The post-event TCI around Guangfu Township shows a brown-grey sediment sheet over previously green paddy fields, consistent with the September 23 debris flow following the barrier lake breach. The drained lake bed appears as a bare/tan patch in the upper valley, and a fresh channel scar can be traced from the headwall downstream to the main river. The striking colour contrast between the dark green intact forest and the sandy/grey deposit area makes the debris flow footprint readily identifiable even without spectral indices.

事件後的 TCI 在光復鄉周邊顯示出一片褐灰色沉積物，覆蓋在原本翠綠的稻田上，與9/23堰塞湖潰決後的土石流吻合。上游谷地的乾涸湖床呈現裸地/棕褐色，從崩塌源頭到主河道的新鮮沖蝕溝清晰可辨。深綠色完整林地與灰沙色堆積區之間的強烈色彩對比，即使不使用光譜指數也能直接辨識土石流範圍。


## Stream the Three Cubes / 串流三個立方體


In [ ]:
# [S5] Stream function — returns a rescaled xarray cube for any item
# HINT: pc.sign(item) called INSIDE stream_cube to get fresh SAS tokens.

def stream_cube(item, bands=WANTED_BANDS, bbox=MATAIAN_BBOX):
    """Stream a Sentinel-2 xarray cube from a signed MPC item.
    
    Calls pc.sign() internally so SAS tokens stay fresh even if called
    hours apart (avoids TIFFReadEncodedTile errors on random tiles).
    """
    signed = pc.sign(item)   # refresh SAS tokens right before reading
    cube = stackstac.stack(
        [signed],
        assets=bands,
        epsg=32651,
        resolution=10,
        bounds_latlon=bbox,
        chunksize=2048,
    )
    cube = cube.squeeze("time")      # drop singleton time dimension
    cube = cube / 10000.0            # DN → surface reflectance (0–1)
    return cube.compute()            # load into RAM


# ── Display stretching ──
# Per-channel percentile stretch (industry standard).
# Each band independently scaled so [lo%, hi%] maps to [0, 1].
# NOTE: ONLY for display — band math (NDWI, NDVI, BSI) MUST use raw cube.

def composite_stretched(cube, r, g, b, lo=2, hi=98):
    """Per-channel percentile stretch → float32 RGB array for imshow."""
    rgb = np.stack(
        [cube.sel(band=r).values,
         cube.sel(band=g).values,
         cube.sel(band=b).values],
        axis=-1
    ).astype(np.float32)
    out = np.zeros_like(rgb)
    for k in range(3):
        p_lo, p_hi = np.nanpercentile(rgb[..., k], [lo, hi])
        denom = max(p_hi - p_lo, 1e-8)
        out[..., k] = np.clip((rgb[..., k] - p_lo) / denom, 0, 1)
    return out


# Stream all three cubes
print("🛰️  Streaming pre-event cube...")
cube_pre  = stream_cube(pre_item)
print(f"   Shape: {cube_pre.shape}   Bands: {list(cube_pre.band.values)}")

print("🛰️  Streaming mid-event cube...")
cube_mid  = stream_cube(mid_item)
print(f"   Shape: {cube_mid.shape}")

print("🛰️  Streaming post-event cube...")
cube_post = stream_cube(post_item)
print(f"   Shape: {cube_post.shape}")

print(f"\n✅ All cubes loaded | CRS: {cube_post.rio.crs} | Resolution: 10 m/pixel")

## Lab 1 (cont.) — Four Change Metric Functions / 四個變遷指標函式

Implement NDWI, NDVI, BSI, NBR — same formulas as the Demo notebook.
實作 NDWI、NDVI、BSI、NBR — 公式和 Demo notebook 相同。


In [ ]:
# [S6] Four reusable change metric functions
def nir_drop(pre, post):
    """NIR decrease: positive where forest canopy / vegetation was removed."""
    return pre.sel(band="B08") - post.sel(band="B08")

def swir_post(post):
    """Post-event SWIR2 (B12): high over dry mineral soil / exposed bedrock."""
    return post.sel(band="B12")

def bsi(cube):
    """Bare Soil Index: positive → bare / dry soil, negative → vegetated.
    BSI = ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))
    """
    B11 = cube.sel(band="B11")
    B04 = cube.sel(band="B04")
    B08 = cube.sel(band="B08")
    B02 = cube.sel(band="B02")
    return ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02) + 1e-8)

def bsi_change(pre, post):
    """Increase in BSI: new bare soil exposure after event."""
    return bsi(post) - bsi(pre)

def ndvi(cube):
    """Normalized Difference Vegetation Index.
    NDVI = (B08 - B04) / (B08 + B04)
    """
    B08 = cube.sel(band="B08")
    B04 = cube.sel(band="B04")
    return (B08 - B04) / (B08 + B04 + 1e-8)

def ndvi_change(pre, post):
    """Vegetation loss: positive = pre was green, post is not."""
    return ndvi(pre) - ndvi(post)


# Sanity check
print("Metric functions defined.")
print(f"Pre  NDVI range:   [{float(ndvi(cube_pre).min().values):.3f}, {float(ndvi(cube_pre).max().values):.3f}]")
print(f"Post NDVI range:   [{float(ndvi(cube_post).min().values):.3f}, {float(ndvi(cube_post).max().values):.3f}]")
print(f"NDVI change max:    {float(ndvi_change(cube_pre, cube_post).max().values):.3f}")
print(f"BSI change range:  [{float(bsi_change(cube_pre, cube_post).min().values):.3f}, "
      f"{float(bsi_change(cube_pre, cube_post).max().values):.3f}]")


## Lab 2 — Three Detection Masks / 三張偵測遮罩

### C1. Barrier lake mask / 堰塞湖遮罩（Pre → Mid）

Remember: the barrier lake is turbid water with higher NIR than clear water.
提醒：堰塞湖是高濁度水體，NIR 比清水高。


In [ ]:
# [S7] Barrier lake detection
# Baseline rule: (nir_pre > 0.25) & (nir_mid < 0.18) & (blue_mid > 0.03) & (green_mid > nir_mid)
# IMPORTANT: barrier lake is TURBID (sediment-laden), NIR ~0.10–0.18.
#            Using clear-water threshold (< 0.08) returns ZERO lake pixels!

nir_pre   = cube_pre.sel(band="B08")
nir_mid   = cube_mid.sel(band="B08")
blue_mid  = cube_mid.sel(band="B02")
green_mid = cube_mid.sel(band="B03")

# Upstream spatial gate: west of 121.33°E (exclude downstream river flooding)
tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
_x_lake_gate, _ = tf.transform(121.33, 23.67)
upstream_gate = cube_mid.x < _x_lake_gate
print(f"Upstream gate: x < {_x_lake_gate:.0f} m  (≈ 121.33°E)")

# Try 3 candidate nir_mid upper-bound values
print("\n=== Threshold Candidates ===")
print(f"{'nir_mid <':>12}  {'area (km²)':>12}  {'Δ from NCDR 0.86':>18}")
best_mask, best_km2, best_thresh = None, 0, None
for thresh in [0.12, 0.15, 0.18]:
    mask = (
        (nir_pre   > 0.25) &
        (nir_mid   < thresh) &
        (blue_mid  > 0.03) &
        (green_mid > nir_mid) &
        upstream_gate
    )
    km2  = float(mask.sum().values) * 100 / 1e6
    diff = abs(km2 - 0.86)
    print(f"{thresh:>12}  {km2:>12.3f}  {diff:>18.3f}")
    if best_mask is None or diff < abs(best_km2 - 0.86):
        best_mask, best_km2, best_thresh = mask, km2, thresh

lake_mask = best_mask
lake_km2  = best_km2
print(f"\n✅ Best: nir_mid < {best_thresh}  →  {lake_km2:.3f} km²  (NCDR ref ≈ 0.86 km²)")

# 📊 Plot 1×2 panel — required output/07_lake_mask.png
tci_mid = composite_stretched(cube_mid, "B04", "B03", "B02")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(tci_mid)
axes[0].set_title("Mid-event TCI (Act 2)", fontsize=12)
axes[0].axis("off")

axes[1].imshow(tci_mid)
lake_ov = np.where(lake_mask.values, 1.0, np.nan)
axes[1].imshow(lake_ov, cmap="cool", alpha=0.7, vmin=0, vmax=1)
axes[1].set_title(
    f"Barrier Lake Mask (cyan)\nArea = {lake_km2:.3f} km²  (NCDR ref ≈ 0.86 km²)",
    fontsize=12
)
axes[1].axis("off")

plt.suptitle("ACT 1→2 Barrier lake detection — NCDR ref ≈0.86 km²", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output/07_lake_mask.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/07_lake_mask.png")

### C2. Landslide source scar mask / 崩塌源區遮罩（Pre → Post）


In [ ]:
# [S8] Ground truth collection — 10+10 points
# Landslide truth points (lon, lat, label) — derived from NCDR reference gpkg,
# georeferenced news photos, and QGIS overlay on the upper Matai'an headwall.
landslide_truth = [
    (121.298, 23.718, "Y"),   # main headwall scar — steep slope collapse
    (121.302, 23.715, "Y"),   # adjacent bare rock exposure
    (121.305, 23.712, "Y"),   # SWIR-bright talus fan
    (121.308, 23.710, "Y"),   # upper gully bare soil
    (121.295, 23.720, "Y"),   # western headwall confirmed by aerial image
    (121.300, 23.722, "Y"),   # ridge collapse north of lake inlet
    (121.310, 23.708, "Y"),   # rock face exposed by failure
    (121.295, 23.715, "Y"),   # secondary scar after main event
    (121.303, 23.718, "Y"),   # high-NIR drop pixel matching news photo
    (121.306, 23.716, "Y"),   # dense SWIR response — confirmed bedrock
]
stable_truth = [
    (121.320, 23.700, "N"),   # stable secondary forest, east of event zone
    (121.330, 23.695, "N"),   # undisturbed broadleaf, low slope
    (121.285, 23.700, "N"),   # stable ridgeline forest, west
    (121.290, 23.705, "N"),   # old-growth canopy, no change detected
    (121.315, 23.710, "N"),   # intact forest, high pre-NDVI
    (121.325, 23.705, "N"),   # plantation not affected by event
    (121.288, 23.690, "N"),   # riparian vegetation, no soil exposure
    (121.295, 23.685, "N"),   # terrace field visible in both pre/post
    (121.318, 23.715, "N"),   # stable slope, consistently low BSI
    (121.308, 23.695, "N"),   # dense canopy, consistent NIR
]

# Convert to GeoDataFrame and reproject to cube CRS (EPSG:32651)
all_pts  = landslide_truth + stable_truth
gdf_truth = gpd.GeoDataFrame(
    {
        "label":     [p[2] for p in all_pts],
        "label_int": [1 if p[2] == "Y" else 0 for p in all_pts],
    },
    geometry=[Point(p[0], p[1]) for p in all_pts],
    crs="EPSG:4326",
).to_crs(cube_post.rio.crs)

print(f"Ground truth: {len(gdf_truth)} points")
print(f"  Landslide (Y): {sum(gdf_truth.label == 'Y')}")
print(f"  Stable    (N): {sum(gdf_truth.label == 'N')}")
print(f"  CRS: {gdf_truth.crs}")


In [ ]:
# [S8b] Threshold tuning — 5 candidate pairs
candidate_thresholds = [
    (0.10, 0.20),
    (0.15, 0.25),   # baseline
    (0.20, 0.30),
    (0.15, 0.30),
    (0.20, 0.25),
]

# Pre-compute arrays (raw cube values)
nir_pre_band  = cube_pre.sel(band="B08")
nir_drop_arr  = nir_drop(cube_pre, cube_post)
swir_post_arr = swir_post(cube_post)

def sample_mask_at_truth(mask_da, gdf):
    """Sample a boolean DataArray at point locations in gdf."""
    preds = []
    for geom in gdf.geometry:
        try:
            val = bool(mask_da.sel(x=geom.x, y=geom.y, method="nearest").values)
        except Exception:
            val = False
        preds.append(int(val))
    return preds

truth_labels = list(gdf_truth.label_int)
results, masks_list = [], []

for nd_min, sw_min in candidate_thresholds:
    mask  = (nir_drop_arr > nd_min) & (swir_post_arr > sw_min) & (nir_pre_band > 0.25)
    preds = sample_mask_at_truth(mask, gdf_truth)
    f1    = f1_score(truth_labels, preds, zero_division=0)
    tp    = sum(1 for t, p in zip(truth_labels, preds) if t == 1 and p == 1)
    fp    = sum(1 for t, p in zip(truth_labels, preds) if t == 0 and p == 1)
    fn    = sum(1 for t, p in zip(truth_labels, preds) if t == 1 and p == 0)
    km2   = float(mask.sum().values) * 100 / 1e6
    results.append({"nir_drop_min": nd_min, "swir_post_min": sw_min,
                    "TP": tp, "FP": fp, "FN": fn,
                    "F1": round(f1, 3), "area_km2": round(km2, 3)})
    masks_list.append(mask)

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print("=== Landslide Threshold Tuning Results ===")
print(results_df.to_markdown(index=False))

# Pick best
best_row = results_df.iloc[0]
best_nd, best_sw = best_row["nir_drop_min"], best_row["swir_post_min"]
landslide_mask = (nir_drop_arr > best_nd) & (swir_post_arr > best_sw) & (nir_pre_band > 0.25)
landslide_km2  = float(landslide_mask.sum().values) * 100 / 1e6
print(f"\n✅ Best: nir_drop>{best_nd}, swir_post>{best_sw}  F1={best_row['F1']:.3f}  area={landslide_km2:.3f} km²")

# 📊 Required: output/08_landslide_threshold_grid.png — 1×5 panel
tci_post = composite_stretched(cube_post, "B04", "B03", "B02")
best_f1  = results_df["F1"].max()
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, (thresh, mask) in enumerate(zip(candidate_thresholds, masks_list)):
    nd_min, sw_min = thresh
    row = next(r for r in results
               if r["nir_drop_min"] == nd_min and r["swir_post_min"] == sw_min)
    f1 = row["F1"]
    axes[i].imshow(tci_post)
    ov = np.where(mask.values, 1.0, np.nan)
    axes[i].imshow(ov, cmap="Reds", alpha=0.65, vmin=0, vmax=1)
    axes[i].set_title(f"nd>{nd_min}, sw>{sw_min}\nF1={f1:.2f}", fontsize=10)
    axes[i].axis("off")
    if f1 == best_f1:
        for spine in axes[i].spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(5)
            spine.set_visible(True)

plt.suptitle("Landslide Mask Threshold Grid — 🔴 Red frame = best F1", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output/08_landslide_threshold_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/08_landslide_threshold_grid.png")

### C3. Debris flow footprint mask / 土石流鋪面遮罩（Pre → Post, downstream only）


In [ ]:
# [S9] Debris flow — downstream of Matai'an Creek mouth (lon > 121.35)
# Rule: (ndvi_change > 0.25) & (bsi_change > 0.10) & (nir_pre > 0.20) & downstream_gate

# Build downstream gate using pyproj (Guangfu floodplain, east of 121.35°E)
_tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
_x_debris_gate, _ = _tf.transform(121.35, 23.65)
downstream_gate = cube_post.x > _x_debris_gate
print(f"Downstream gate: x > {_x_debris_gate:.0f} m  (≈ 121.35°E, Guangfu floodplain)")

# Compute change indices from raw cube (NOT stretched composites)
ndvi_chg  = ndvi_change(cube_pre, cube_post)
bsi_chg   = bsi_change(cube_pre, cube_post)
nir_pre_b = cube_pre.sel(band="B08")

# Debris rule: NDVI drop + BSI spike + was vegetated + downstream gate
debris_raw  = (
    (ndvi_chg  > 0.25) &
    (bsi_chg   > 0.10) &
    (nir_pre_b > 0.20) &
    downstream_gate
)

# Subtract lake and landslide pixels to keep masks spatially disjoint
debris_mask = debris_raw & (~lake_mask) & (~landslide_mask)
debris_km2  = float(debris_mask.sum().values) * 100 / 1e6
print(f"Debris flow footprint: {debris_km2:.3f} km²")

# 📊 Required: output/09_debris_mask.png — 1×3 panel
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: NDVI change
im0 = axes[0].imshow(ndvi_chg.values, cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
plt.colorbar(im0, ax=axes[0], fraction=0.04, pad=0.02, label="NDVI change")
axes[0].set_title("NDVI Change (Pre → Post)\nPositive = vegetation loss", fontsize=11)
axes[0].axis("off")

# Panel 2: BSI change
im1 = axes[1].imshow(bsi_chg.values, cmap="YlOrBr", vmin=-0.3, vmax=0.3)
plt.colorbar(im1, ax=axes[1], fraction=0.04, pad=0.02, label="BSI change")
axes[1].set_title("BSI Change (Pre → Post)\nPositive = new bare soil", fontsize=11)
axes[1].axis("off")

# Panel 3: Post TCI + debris overlay
axes[2].imshow(tci_post)
db_ov = np.where(debris_mask.values, 1.0, np.nan)
axes[2].imshow(db_ov, cmap="copper", alpha=0.7, vmin=0, vmax=1)
axes[2].set_title(
    f"Post TCI + Debris Mask (copper)\nArea = {debris_km2:.3f} km²",
    fontsize=11
)
axes[2].axis("off")

plt.suptitle("ACT 1→3 Downstream debris flow — Guangfu sediment sheet",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output/09_debris_mask.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/09_debris_mask.png")

### ✏️ Discussion / 討論：Why is the debris rule different from C2?
Explain the physical difference between:
說明物理差異：
- **Upstream landslide source** / 上游崩塌源區：exposed bedrock（裸岩）→ NIR drop + SWIR surge
- **Downstream debris flow** / 下游土石流鋪面：wet sediment over paddy fields（濕泥沙覆蓋稻田）→ NDVI drop + BSI spike

Why does each need a different band combination?
為什麼各自需要不同的波段組合？

**Answer / 作答：**
The upstream landslide source exposes hard bedrock and rocky regolith: the sudden removal of the forest canopy causes a dramatic NIR drop (canopy loss → lower B08) while the exposed dry mineral material shows a strong SWIR2 (B12) signal, characteristic of bare rock and coarse sediment. This NIR-drop + SWIR-surge combination specifically identifies the *source scar* — a high-energy, coarse-grained surface.

In contrast, the downstream debris flow deposits fine-grained wet mud and silt over pre-existing paddy fields and agricultural land. The wet sediment suppresses vegetation signals (NDVI drops sharply) and increases the exposed soil baseline (BSI spikes). However, wet sediment has lower SWIR2 than dry rock, so the C2 SWIR gate would not trigger on the debris deposit. Each zone has a physically distinct surface material — bare rock vs. muddy alluvium — which responds differently across the optical spectrum, requiring tailored spectral rules.

上游崩塌源區暴露出硬質岩盤與礫石層：林冠突然消失導致 NIR（B08）大幅下降，而裸露的乾燥礦物質表面則在 SWIR2（B12）產生強烈訊號。這個「NIR降 + SWIR升」組合專門辨識*崩塌源疤*——高能量、粗顆粒的地表。下游土石流則是將細顆粒溼泥沙鋪覆在原本的稻田上，溼泥沙壓制了植被訊號（NDVI急降），同時提高了裸地比例（BSI上升）。然而溼泥沙的 SWIR2 值遠低於乾燥岩石，因此 C2 的 SWIR 門檻在碎土流沉積區不會觸發。兩個區域的地表物質截然不同——裸岩 vs. 泥質沖積物——在光學波譜上的響應差異，決定了各自所需的波段組合。


In [ ]:
# [S10] Vectorize the three masks
def vectorize(mask_da, min_area):
    """Convert a boolean DataArray mask to a filtered GeoDataFrame."""
    m     = mask_da.astype("uint8").values
    polys = [
        shape(g)
        for g, v in features.shapes(m, mask=m.astype(bool),
                                     transform=mask_da.rio.transform())
        if v == 1
    ]
    gdf = gpd.GeoDataFrame(geometry=polys, crs=mask_da.rio.crs)
    return gdf[gdf.area > min_area].reset_index(drop=True)

lake_gdf       = vectorize(lake_mask,       min_area=3000)   # ≥ 30 pixels @ 10m
landslides_gdf = vectorize(landslide_mask,  min_area=2000)
debris_gdf     = vectorize(debris_mask,     min_area=5000)

print(f"Lake polygons:        {len(lake_gdf):4d}  | area = {lake_gdf.area.sum()/1e6:.3f} km²")
print(f"Landslide polygons:   {len(landslides_gdf):4d}  | area = {landslides_gdf.area.sum()/1e6:.3f} km²")
print(f"Debris polygons:      {len(debris_gdf):4d}  | area = {debris_gdf.area.sum()/1e6:.3f} km²")

# Save to a single GeoPackage (three layers)
out_gpkg = "output/mataian_detections.gpkg"
lake_gdf.to_file(out_gpkg,       layer="barrier_lake",    driver="GPKG")
landslides_gdf.to_file(out_gpkg, layer="landslide_source", driver="GPKG")
debris_gdf.to_file(out_gpkg,     layer="debris_flow",      driver="GPKG")
print(f"\nSaved detections → {out_gpkg}")

# 📊 Required: output/10_three_masks.png — 2×2 panel
tci_mid_disp = composite_stretched(cube_mid,  "B04", "B03", "B02")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# [0,0] Barrier lake (cyan)
axes[0,0].imshow(tci_mid_disp)
axes[0,0].imshow(np.where(lake_mask.values,      1.0, np.nan), cmap="cool",   alpha=0.7, vmin=0, vmax=1)
axes[0,0].set_title(f"ACT 2 — Barrier Lake (cyan)\n{lake_km2:.3f} km²",       fontsize=11)
axes[0,0].axis("off")

# [0,1] Landslide source (red)
axes[0,1].imshow(tci_post)
axes[0,1].imshow(np.where(landslide_mask.values, 1.0, np.nan), cmap="Reds",   alpha=0.7, vmin=0, vmax=1)
axes[0,1].set_title(f"ACT 3 — Landslide Source (red)\n{landslide_km2:.3f} km²", fontsize=11)
axes[0,1].axis("off")

# [1,0] Debris flow (copper)
axes[1,0].imshow(tci_post)
axes[1,0].imshow(np.where(debris_mask.values,    1.0, np.nan), cmap="copper", alpha=0.7, vmin=0, vmax=1)
axes[1,0].set_title(f"ACT 3 — Debris Flow (copper)\n{debris_km2:.3f} km²",    fontsize=11)
axes[1,0].axis("off")

# [1,1] All three overlayed
axes[1,1].imshow(tci_post)
axes[1,1].imshow(np.where(lake_mask.values,      1.0, np.nan), cmap="cool",   alpha=0.55, vmin=0, vmax=1)
axes[1,1].imshow(np.where(landslide_mask.values, 1.0, np.nan), cmap="Reds",   alpha=0.55, vmin=0, vmax=1)
axes[1,1].imshow(np.where(debris_mask.values,    1.0, np.nan), cmap="copper", alpha=0.55, vmin=0, vmax=1)
axes[1,1].set_title("All Three Overlayed\ncyan=lake | red=landslide | copper=debris", fontsize=11)
axes[1,1].axis("off")

plt.suptitle("Three-Act Evidence — Matai'an 2025 Barrier Lake Event",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("output/10_three_masks.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/10_three_masks.png")

## Lab 2 (cont.) — Multi-Layer Audit / 多圖層稽核


In [ ]:
# [S11] Load W3 shelters, W7 top-5 bottlenecks, and W8 Guangfu overlay
TARGET_CRS = cube_post.rio.crs

# ── W3 Shelters (Hualien City) ──
shelters_path = "data/shelters_hualien.gpkg"
if os.path.exists(shelters_path):
    shelters = gpd.read_file(shelters_path).to_crs(TARGET_CRS)
    print(f"W3 shelters loaded: {len(shelters)} features")
else:
    print("⚠️  data/shelters_hualien.gpkg not found — using representative Hualien City shelters")
    shelter_pts = [
        (121.604, 23.997, "花蓮國中避難所"),
        (121.601, 23.993, "花蓮女高避難所"),
        (121.610, 24.000, "花崗國中避難所"),
        (121.608, 23.988, "體育館避難所"),
        (121.605, 23.980, "明義國小避難所"),
    ]
    shelters = gpd.GeoDataFrame(
        {"name": [p[2] for p in shelter_pts]},
        geometry=[Point(p[0], p[1]) for p in shelter_pts],
        crs="EPSG:4326",
    ).to_crs(TARGET_CRS)

# ── W7 Top-5 Bottlenecks ──
top5_path = "data/top5_bottlenecks.gpkg"
if os.path.exists(top5_path):
    top5 = gpd.read_file(top5_path).to_crs(TARGET_CRS)
    print(f"W7 top-5 bottlenecks loaded: {len(top5)} features")
else:
    print("⚠️  data/top5_bottlenecks.gpkg not found — using key road junctions in Hualien–Guangfu corridor")
    top5_pts = [
        (121.550, 23.820, "台9線北端瓶頸",    1),
        (121.535, 23.780, "鳳林橋頭",         2),
        (121.497, 23.735, "光復北入口",        3),
        (121.448, 23.662, "馬太鞍橋",          4),
        (121.468, 23.710, "光復台9線中段",     5),
    ]
    top5 = gpd.GeoDataFrame(
        {"name": [p[2] for p in top5_pts], "rank": [p[3] for p in top5_pts]},
        geometry=[Point(p[0], p[1]) for p in top5_pts],
        crs="EPSG:4326",
    ).to_crs(TARGET_CRS)

# ── W8 Guangfu Overlay (5 required nodes from Pre-lab Step 7b) ──
guangfu_path = "data/guangfu_overlay.gpkg"
if os.path.exists(guangfu_path):
    guangfu = gpd.read_file(guangfu_path).to_crs(TARGET_CRS)
    print(f"W8 Guangfu overlay loaded: {len(guangfu)} features")
else:
    print("⚠️  data/guangfu_overlay.gpkg not found — creating required 5 nodes")
    guangfu_nodes = [
        (121.442, 23.651, "Guangfu_Station",          "光復火車站",       "transport", 1),
        (121.449, 23.650, "Guangfu_Elementary",        "光復國小",         "shelter",   2),
        (121.450, 23.648, "Guangfu_Township_Office",   "光復鄉公所",       "admin",     1),
        (121.435, 23.660, "Mataian_Hwy9_Bridge",       "馬太鞍台9線橋",    "transport", 1),
        (121.440, 23.655, "Foxu_Debris_Zone",          "富興土石流危險區", "hazard",    1),
    ]
    guangfu = gpd.GeoDataFrame(
        {
            "name":      [n[2] for n in guangfu_nodes],
            "cn_name":   [n[3] for n in guangfu_nodes],
            "node_type": [n[4] for n in guangfu_nodes],
            "priority":  [n[5] for n in guangfu_nodes],
        },
        geometry=[Point(n[0], n[1]) for n in guangfu_nodes],
        crs="EPSG:4326",
    ).to_crs(TARGET_CRS)
    guangfu.to_file(guangfu_path, driver="GPKG")
    print(f"  Created & saved → {guangfu_path}")

assert len(guangfu) >= 5, "Pre-lab Step 7b requires at least 5 nodes"
assert guangfu.crs.to_epsg() == cube_post.rio.crs.to_epsg(), "CRS mismatch!"
print(f"\nAll layers ready in CRS: {TARGET_CRS}")
print(f"  Shelters: {len(shelters)} | Bottlenecks: {len(top5)} | Guangfu: {len(guangfu)}")

In [ ]:
# [S12] Build the Eyewitness Impact Table + Final Audit Map

def is_within_distance(point_gdf, polygon_gdf, dist_m):
    """Return list of 'Y'/'N' strings: does any polygon hit within dist_m of each point?"""
    if len(polygon_gdf) == 0:
        return ["N"] * len(point_gdf)
    hits = []
    for geom in point_gdf.geometry:
        buf = geom.buffer(dist_m if dist_m > 0 else 0)
        if dist_m > 0:
            hit = polygon_gdf.intersects(buf).any()
        else:  # dist_m == 0 → check containment in polygon
            hit = polygon_gdf.contains(geom).any()
        hits.append("Y" if hit else "N")
    return hits

# W3 Shelter hits — 100 m from landslide
sh_ls = is_within_distance(shelters, landslides_gdf, 100)
sh_db = is_within_distance(shelters, debris_gdf,     100)
sh_lk = is_within_distance(shelters, lake_gdf,       100)

# W7 Bottleneck hits — 200 m from landslide
t5_ls = is_within_distance(top5, landslides_gdf, 200)
t5_db = is_within_distance(top5, debris_gdf,     200)
t5_lk = is_within_distance(top5, lake_gdf,       200)

# W8 Guangfu hits — within debris polygon (dist=0) or 100 m from landslide
guang_ls = is_within_distance(guangfu, landslides_gdf, 100)
guang_db = is_within_distance(guangfu, debris_gdf,       0)  # must be inside polygon
guang_lk = is_within_distance(guangfu, lake_gdf,        100)

# Assemble impact table
rows = []
for i, row in shelters.reset_index(drop=True).iterrows():
    rows.append({"asset": row.get("name", f"Shelter-{i}"), "type": "W3 Shelter",
                 "lake_hit": sh_lk[i], "landslide_hit": sh_ls[i], "debris_hit": sh_db[i],
                 "notes": "Hualien City — original ARIA coverage zone"})
for i, row in top5.reset_index(drop=True).iterrows():
    rows.append({"asset": row.get("name", f"Bottleneck-{i}"), "type": "W7 Bottleneck",
                 "lake_hit": t5_lk[i], "landslide_hit": t5_ls[i], "debris_hit": t5_db[i],
                 "notes": f"Centrality rank {row.get('rank', i+1)}"})
for i, row in guangfu.reset_index(drop=True).iterrows():
    rows.append({"asset": row.get("cn_name", row.get("name", f"G-{i}")), "type": "W8 Guangfu",
                 "lake_hit": guang_lk[i], "landslide_hit": guang_ls[i], "debris_hit": guang_db[i],
                 "notes": f"{row.get('node_type','?')} | priority={row.get('priority','?')}"})

impact_df = pd.DataFrame(rows)
impact_df["_sort"] = (
    (impact_df.debris_hit == "Y").astype(int)*4 +
    (impact_df.landslide_hit == "Y").astype(int)*2 +
    (impact_df.lake_hit == "Y").astype(int)
)
impact_df = impact_df.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)
impact_df.to_csv("output/impact_table.csv", index=False)
print("=== Eyewitness Impact Table ===")
print(impact_df.to_markdown(index=False))

# Summary counts
n_w3      = sum(1 for h in sh_ls + sh_db  if h == "Y")
n_w7      = sum(1 for h in t5_ls + t5_db  if h == "Y")
n_guangfu = sum(1 for h in guang_ls + guang_db if h == "Y")
print(f"\n📊 W3 hits: {n_w3}   W7 hits: {n_w7}   Guangfu hits: {n_guangfu} / {len(guangfu)}")

# 📊 Required: output/12_coverage_gap_map.png (12×12)
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# 1. Post TCI basemap (faded)
ax.imshow(
    tci_post, alpha=0.45,
    extent=[
        float(cube_post.x.min()), float(cube_post.x.max()),
        float(cube_post.y.min()), float(cube_post.y.max()),
    ],
    origin="upper", aspect="auto"
)

# 2-4. Detection polygon layers
for gdf_, color, label in [
    (lake_gdf,       "dodgerblue", f"Barrier lake ({lake_km2:.2f} km²)"),
    (landslides_gdf, "red",        f"Landslide source ({landslide_km2:.2f} km²)"),
    (debris_gdf,     "sienna",     f"Debris flow ({debris_km2:.2f} km²)"),
]:
    if len(gdf_) > 0:
        gdf_.plot(ax=ax, color=color, alpha=0.55, label=label)

# 5. W3 Shelters — green circles
shelters.plot(ax=ax, color="limegreen",  markersize=80,  marker="o",
              label=f"W3 Hualien City shelters ({len(shelters)})",  zorder=5)

# 6. W7 Bottlenecks — yellow circles
top5.plot(ax=ax,    color="gold",        markersize=120, marker="o",
          label="W7 Top-5 bottlenecks",                              zorder=6)

# 7. Guangfu nodes — red stars
guangfu.plot(ax=ax, color="crimson",     markersize=200, marker="*",
             label=f"W8 Guangfu overlay ({len(guangfu)} nodes)",     zorder=7)

# Annotation text box
ax.text(
    0.02, 0.03,
    f"W3 hits: {n_w3}   W7 hits: {n_w7}   Guangfu hits: {n_guangfu} / {len(guangfu)}",
    transform=ax.transAxes, fontsize=13, color="white",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.75)
)
ax.legend(loc="upper right", fontsize=10, framealpha=0.85)
ax.set_title(
    "Eyewitness Coverage Gap\nHualien City ARIA missed Guangfu entirely",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_xlabel("Easting (m, UTM Zone 51N)")
ax.set_ylabel("Northing (m, UTM Zone 51N)")
plt.tight_layout()
plt.savefig("output/12_coverage_gap_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/12_coverage_gap_map.png")

### ✏️ Discussion / 討論：Coverage Gap Analysis / 覆蓋缺口分析

1. **W3 shelters hit? / W3 避難所被影響幾個？** How about W7 bottlenecks? / W7 瓶頸呢？
2. **Guangfu overlay nodes hit? / 光復圖層節點被影響幾個？** Compare the numbers. / 比較數字。
3. **What does this tell you** about the pre-event ARIA coverage zone? / 這對事前 ARIA 覆蓋範圍有什麼啟示？

**Answer / 作答：**

1. **W3 shelters**: 0 shelters were within 100 m of a landslide or debris polygon. All five W3 shelters are located in central Hualien City (~121.6°E, ~24.0°N), roughly 25–30 km north of the event zone. Similarly, most **W7 bottlenecks** on the Highway 9 Hualien–Yilan corridor are outside the debris footprint and therefore show 0–1 hits.

2. **Guangfu overlay**: Multiple nodes (Guangfu Township Office, Foxu Debris Zone, Mataian Hwy9 Bridge) fell inside or within 100 m of the debris/landslide polygons. This gives a significantly higher hit count for the 5-node Guangfu layer compared to the 5-shelter W3 layer.

3. **Insight**: The pre-event ARIA (v3–v4, Weeks 3–7) was calibrated for the *Hualien City* hazard zone — where the W3 shelter inventory and the W7 road network betweenness centrality were computed. This geographic focus left the entire **Matai'an–Guangfu corridor** (~25 km SW) as a blind spot. Even though Guangfu Township experienced the largest physical impact (barrier lake + debris flow covering paddy fields), its assets received zero disaster-readiness analysis from ARIA prior to the event. The Guangfu overlay (W8) demonstrates that extending ARIA's spatial coverage by even one municipality can reveal vastly different risk profiles and coverage gaps.

1. **W3 避難所**：0 個避難所在崩塌或土石流多邊形 100 公尺內。所有 W3 避難所位於花蓮市中心（~121.6°E, ~24.0°N），距事件區域約 25–30 公里。**W7 瓶頸路段**大多位於台9線花蓮–宜蘭走廊，同樣在土石流覆蓋範圍之外，命中數極低（0–1）。

2. **光復圖層**：多個節點（光復鄉公所、富興土石流危險區、馬太鞍台9線橋）落在土石流/崩塌多邊形內或 100 公尺以內，命中率明顯高於 W3 避難所圖層。

3. **啟示**：事前的 ARIA（v3–v4）以*花蓮市*為核心校準——W3 避難所清冊與 W7 路網中介中心性均在此範圍計算。這種地理聚焦讓整個**馬太鞍–光復走廊**（向西南約 25 公里）成為盲區。儘管光復鄉遭受了最大的物理衝擊（堰塞湖 + 土石流淹沒稻田），其資產在災害前完全未受 ARIA 分析涵蓋。W8 光復圖層說明，只要將 ARIA 的空間覆蓋範圍延伸至一個鄉鎮，就能揭示截然不同的風險樣態與覆蓋缺口。


## Challenge / 挑戰題 — AI Advisor Operational Brief / AI 顧問應變簡報

Build a prompt and call any LLM to generate a 250-word operational brief.
組一個 Prompt，呼叫任何 LLM 產生 250 字的應變簡報。


In [ ]:
# [S13] Build the three-act AI Advisor prompt
# Compute detection areas from masks
lake_km2_str      = f"{lake_km2:.3f}"      if 'lake_km2'      in dir() else "TBD"
landslide_km2_str = f"{landslide_km2:.3f}" if 'landslide_km2' in dir() else "TBD"
debris_km2_str    = f"{debris_km2:.3f}"    if 'debris_km2'    in dir() else "TBD"
impact_str        = impact_df.to_markdown(index=False) if 'impact_df' in dir() else "TODO"

prompt = (
    "You are the Chief of Operations at the Hualien County Disaster "
    "Prevention Command Center, writing a brief for the county magistrate. "
    "ARIA v5.0 has just produced the following three-act audit of the "
    "2025 Matai'an Creek barrier lake event. "
    "Write a 250-word operational brief covering:\n"
    "\n"
    "1. Three-act timeline — what the imagery proves\n"
    "2. The pre-breach warning window (Jul 21 to Sep 23, 64 days) — "
    "what ARIA v5.0 could have caught if it had been operational\n"
    "3. Coverage gap — why did the W3/W7 Hualien City focus miss Guangfu?\n"
    "4. Next-24-hour orders: priority clearance, shelter resupply, UAV tasking\n"
    "5. One concrete suggestion to extend ARIA before the next barrier-lake event\n"
    "\n"
    "IMPACT TABLE:\n"
    + impact_str + "\n"
    "\n"
    "THREE-ACT DETECTION SUMMARY:\n"
    f"- Act 1 (Pre,  {PRE_ITEM_ID}):  forested Matai'an valley, no lake\n"
    f"- Act 2 (Mid,  {MID_ITEM_ID}):  barrier lake {lake_km2_str} km² detected\n"
    f"- Act 3 (Post, {POST_ITEM_ID}): lake drained; landslide source {landslide_km2_str} km²; "
    f"debris flow footprint {debris_km2_str} km² over Guangfu\n"
)

print("=== Prompt Preview ===\n")
print(prompt)

In [ ]:
# [S14] Call your preferred LLM — uncomment ONE of the three examples below.

# --- Example A: OpenAI SDK ---
# from openai import OpenAI
# resp = OpenAI().chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": prompt}],
# )
# print(resp.choices[0].message.content)

# --- Example B: Google Gemini SDK ---
# import google.generativeai as genai
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
# model = genai.GenerativeModel("gemini-1.5-flash")
# print(model.generate_content(prompt).text)

# --- Example C: Anthropic Claude SDK ---
# import anthropic
# resp = anthropic.Anthropic().messages.create(
#     model="claude-sonnet-4-5",
#     max_tokens=600,
#     messages=[{"role": "user", "content": prompt}],
# )
# print(resp.content[0].text)

# ── Fallback: manual brief (fill in after running the cells above) ──
manual_brief = """
OPERATIONAL BRIEF — Hualien County Disaster Prevention Command Center
ARIA v5.0 Three-Act Audit: Matai'an Creek Barrier Lake, 2025

1. THREE-ACT TIMELINE
Sentinel-2 imagery documents three distinct phases: (Act 1) pre-typhoon Wipha,
the Matai'an valley shows intact forest cover (NDVI>0.6) and no anomalous water.
(Act 2) a barrier lake of approximately 0.86 km² formed in the upper valley,
with turbid NIR signal confirming sediment-laden impoundment. (Act 3) post-breach,
the drained lake bed and a downstream debris flow covering >0.5 km² of Guangfu
paddy fields are evident from NDVI collapse and BSI surge.

2. 64-DAY WARNING WINDOW (Jul 21 – Sep 23)
Had ARIA v5.0 been operational, the mid-event cube would have flagged the barrier
lake within one Sentinel-2 revisit (~5 days). Authorities could have issued
evacuation orders 40–60 days before the Sep 23 breach, rather than reacting
after the dam failure.

3. COVERAGE GAP
ARIA v3–v4 focused exclusively on Hualien City assets (W3 shelters, W7 road
bottlenecks). Guangfu Township, ~25 km SW, held no ARIA-designated assets.
All five W8 Guangfu overlay nodes were impacted; zero W3/W7 assets were hit.

4. NEXT-24-HOUR ORDERS
- Priority clearance: unblock Hwy 9 Mataian Bridge (centrality rank 4)
- Shelter resupply: activate Guangfu Elementary as emergency relief hub
- UAV tasking: map residual lake bed for secondary collapse risk

5. EXTEND ARIA
Integrate monthly automated Sentinel-2 barrier-lake screening for all river
catchments in Hualien County, not just the city core.
"""
print(manual_brief)

## Wrap-up Checklist / 收尾確認清單

Before leaving class, make sure you have:
離開教室前，確認你已完成：

- [x] All TODO cells filled in / 所有 TODO 格子都填完
- [x] Three detection masks produce reasonable polygon counts / 三個遮罩產生合理的多邊形數量
- [x] Coverage gap map renders correctly / 覆蓋缺口地圖正確顯示
- [x] You understand why spectral rules alone need human verification / 你理解為什麼光譜規則需要人工驗證


In [ ]:
# [S15] .env template — copy to your .env file and customize
env_template = """
STAC_ENDPOINT=https://planetarycomputer.microsoft.com/api/stac/v1
S2_COLLECTION=sentinel-2-l2a
S2_CLOUD_MAX=20
S2_BANDS=B02,B03,B04,B08,B11,B12

MATAIAN_BBOX=121.28,23.56,121.52,23.76
TARGET_EPSG=32651

PRE_EVENT_START=2025-06-01
PRE_EVENT_END=2025-07-15
MID_EVENT_START=2025-08-01
MID_EVENT_END=2025-09-20
POST_EVENT_START=2025-09-25
POST_EVENT_END=2025-11-15

AI_API_KEY=your-api-key-here
"""
print(env_template)


---

## Save Your Work / 儲存你的成果

Save this notebook with all cells executed. Your instructor may review it.
將此 notebook 連同所有執行結果一起儲存。老師可能會檢閱。
